In [ ]:
#@title Import Library & Download Dataset

!pip install roboflow

from roboflow import Roboflow
import os
import hashlib
import joblib
import cv2
import numpy as np
import pandas as pd
from skimage.feature import hog
from sklearn.preprocessing import StandardScaler
from google.colab import userdata

api_key = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=api_key)
project = rf.workspace("brad-dwyer").project("pothole-voxrl")
version = project.version(1)
dataset = version.download("tensorflow")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 8.1 MB/s  0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.3
    Uninstalling typer-0.27.3:
      Successfully uninstalled typer-0.27.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [roboflow]
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Pothole-1 in tensorflow:: 100%|██████████| 673/673 [00:00<00:00, 4168.36it/s]


In [ ]:
#@title Setup Direktori Output

np.random.seed(42)

BASE_DIR = "../content/Pothole-1"
PROCESSED_DIR = "../content/data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [ ]:
#@title Data Cleaning, Deduplikasi & Target Derivation

splits = ["train", "valid", "test"]
df_list = []

for split in splits:
    csv_path = os.path.join(BASE_DIR, split, "_annotations.csv")
    temp = pd.read_csv(csv_path)
    temp["split"] = split
    temp["filepath"] = temp["filename"].apply(
        lambda fname: os.path.join(BASE_DIR, split, fname)
    )
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

# 1. Hapus missing values & baris dengan file tidak ditemukan
df = df.dropna().copy()
df = df[df["filepath"].apply(os.path.exists)]

# 2. Filter koordinat bounding box valid & clipping ke batas resolusi citra
df = df[(df["xmax"] > df["xmin"]) & (df["ymax"] > df["ymin"]) & (df["width"] > 0) & (df["height"] > 0)]
df["xmin"] = df["xmin"].clip(lower=0)
df["ymin"] = df["ymin"].clip(lower=0)
df["xmax"] = np.minimum(df["xmax"], df["width"])
df["ymax"] = np.minimum(df["ymax"], df["height"])

# 3. Hitung Area Ratio per bounding box
df["area_ratio"] = ((df["xmax"] - df["xmin"]) * (df["ymax"] - df["ymin"])) / (df["width"] * df["height"])

# 4. Agregasi tingkat citra
df_img = (
    df.groupby(["filepath", "filename", "split"], as_index=False)
    .agg(
        pothole_count=("class", "count"),
        max_area_ratio=("area_ratio", "max"),
        total_area_ratio=("area_ratio", "sum")
    )
)

# 5. Deduplikasi MD5 (prioritaskan train -> valid -> test agar test/valid bersih dari duplikat)
def get_md5(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

df_img["md5"] = df_img["filepath"].apply(get_md5)
split_priority = {"train": 0, "valid": 1, "test": 2}
df_img["split_rank"] = df_img["split"].map(split_priority)
df_img = (
    df_img.sort_values("split_rank")
    .drop_duplicates(subset=["md5"], keep="first")
    .drop(columns=["split_rank"])
)

# 6. Mapping Severity Label & Label Encoding
def assign_severity(ratio):
    if ratio < 0.02:
        return "Low"
    elif ratio <= 0.07:
        return "Medium"
    else:
        return "Critical"

label_map = {"Low": 0, "Medium": 1, "Critical": 2}
df_img["severity_label"] = df_img["max_area_ratio"].apply(assign_severity)
df_img["target"] = df_img["severity_label"].map(label_map)

print(f"Total citra bersih siap pakai: {len(df_img)}")

Total citra bersih siap pakai: 665


In [ ]:
#@title Pemisahan Partisi Bawaan Roboflow & Simpan Metadata

train_df = df_img[df_img["split"] == "train"].reset_index(drop=True)
val_df   = df_img[df_img["split"] == "valid"].reset_index(drop=True)
test_df  = df_img[df_img["split"] == "test"].reset_index(drop=True)

print(f"Train : {len(train_df)} citra")
print(f"Valid : {len(val_df)} citra")
print(f"Test  : {len(test_df)} citra")

# Simpan metadata CSV untuk digunakan kembali oleh Model 3 (Transfer Learning CNN)
train_df.to_csv(os.path.join(PROCESSED_DIR, "train_meta.csv"), index=False)
val_df.to_csv(os.path.join(PROCESSED_DIR, "val_meta.csv"), index=False)
test_df.to_csv(os.path.join(PROCESSED_DIR, "test_meta.csv"), index=False)

Train : 465 citra
Valid : 133 citra
Test  : 67 citra


In [ ]:
#@title Ekstraksi Fitur Color Histogram + HOG

def extract_classical_features(df_subset, img_size=(224, 224)):
    feature_list = []
    for path in df_subset["filepath"]:
        img = cv2.imread(path)
        img_resized = cv2.resize(img, img_size)

        # 1. Color Histogram RGB (32 bins x 3 channel = 96 fitur)
        color_hist = []
        for ch in range(3):
            hist = cv2.calcHist([img_resized], [ch], None, [32], [0, 256])
            cv2.normalize(hist, hist)
            color_hist.extend(hist.flatten())

        # 2. HOG Features pada citra Grayscale
        gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
        hog_features = hog(
            gray,
            orientations=9,
            pixels_per_cell=(16, 16),
            cells_per_block=(2, 2),
            block_norm="L2-Hys",
            feature_vector=True
        )

        combined_vector = np.hstack([color_hist, hog_features])
        feature_list.append(combined_vector)

    return np.array(feature_list, dtype=np.float32)

print("Mengekstrak fitur Color Histogram + HOG...")
X_train_raw = extract_classical_features(train_df)
X_val_raw   = extract_classical_features(val_df)
X_test_raw  = extract_classical_features(test_df)

y_train = train_df["target"].values
y_val   = val_df["target"].values
y_test  = test_df["target"].values

print("Dimensi fitur mentah (X_train):", X_train_raw.shape)

Mengekstrak fitur Color Histogram + HOG...
Dimensi fitur mentah (X_train): (465, 6180)


In [ ]:
#@title Scaling Data & Menyimpan Artefak (menyimpan ekstraksi fitur)

# Fit StandardScaler HANYA pada data training
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val   = scaler.transform(X_val_raw)
X_test  = scaler.transform(X_test_raw)

# Simpan dataset terproses & scaler ke data/processed/
np.savez_compressed(
    os.path.join(PROCESSED_DIR, "classical_features.npz"),
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    X_test=X_test, y_test=y_test
)
joblib.dump(scaler, os.path.join(PROCESSED_DIR, "standard_scaler.joblib"))

print("Selesai! Artefak tersimpan di:", PROCESSED_DIR)

Selesai! Artefak tersimpan di: ../content/data/processed
